# How reliable is *local* validation here? (a measurement)

**TL;DR — for ranking post-processing configs, offline scores can *invert* the leaderboard.** So don't
trust local CV to choose between configs; the public LB is the arbiter. Below: two reproducible checks
(run on the competition data) and one measurement (four configs, LB vs offline) that shows the inversion.

This builds on @leewhieldon's observation that the visible test clips are train placeholders — the useful
follow-up he raised (*"how should we interpret local validation then?"*) never got a quantitative answer, so
here's one.


## 1. The visible test clips are train placeholders (so any CV must use held-out *train*)

The 4 clips in `test/` are byte-identical copies of `train/` clips, **with** ground-truth `.geff`. They exist
only to check your notebook emits a well-formed `submission.csv`; at grading a *different, hidden* test set is
substituted (that's why internet is off in submission mode). Quick verification:

In [ ]:
import hashlib, json
from pathlib import Path
CANDS = [Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
         Path("/kaggle/input/biohub-cell-tracking-during-development")]
COMP = next((p for p in CANDS if p.exists()), CANDS[0])
TRAIN, TEST = COMP/"train", COMP/"test"

test_ids = sorted(p.stem for p in TEST.glob("*.zarr"))
print(f"visible test clips: {test_ids}\n")
for cid in test_ids:
    trz, tz = TRAIN/f"{cid}.zarr", TEST/f"{cid}.zarr"
    same, checked = True, 0
    for t in (0, 50, 99):
        a, b = tz/f"0/c/{t}/0/0/0", trz/f"0/c/{t}/0/0/0"
        if a.exists() and b.exists():
            checked += 1
            if hashlib.md5(a.read_bytes()).hexdigest() != hashlib.md5(b.read_bytes()).hexdigest():
                same = False
    print(f"  {cid}: {checked} chunks byte-identical to train = {same} | train GT present = {(TRAIN/f'{cid}.geff').exists()}")
print("\n=> local scoring on the visible test/ folder is meaningless; you must hold out TRAIN movies.")


## 2. The detection GT is *sparse* (so even a held-out-train score is noisy + optimistic)

Each movie's `.geff` carries an `estimated_number_of_nodes` (the true cell count) alongside the far smaller
set of *labeled* nodes. Only a small fraction of nuclei/lineages are annotated:

In [ ]:
import numpy as np
pct = []
for g in sorted(TRAIN.glob("*.geff")):
    try:
        n_est = json.load(open(g/"zarr.json")).get("attributes",{}).get("geff",{}).get("extra",{}).get("estimated_number_of_nodes")
        idp = g/"nodes"/"ids"/"zarr.json"
        n_lab = json.load(open(idp)).get("shape",[None])[0] if idp.exists() else None
        if n_est and n_lab: pct.append(100*n_lab/n_est)
    except Exception:
        pass
pct = np.array(pct)
print(f"movies: {len(pct)} | labeled nuclei: median {np.median(pct):.1f}%  (min {pct.min():.1f}%, max {pct.max():.1f}%)")
print("=> tiny, biologically-selected label sets -> local edge-jaccard is high-variance and over-rewards node-dropping.")


## 3. The measurement: offline ranking **inverts** the LB

We took **four post-processing variants (A–D)** of the same public pipeline — *identical detector*, differing
only in graph post-processing (progressively raising the ILP appearance/disappearance cost and division-recovery
aggressiveness). Their **public-LB scores increase monotonically A→D**. We then scored the *same four configs*
**offline**: on a leak-free held-out split of the train movies, with the community-reconstructed metric
(`adj_edge_jaccard`). The offline ranking comes out **backwards**.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr
cfg     = ["A", "B", "C", "D"]
lb      = [0.901, 0.905, 0.907, 0.908]     # public leaderboard (monotone UP)
offline = [0.9626, 0.9613, 0.9611, 0.9611] # leak-free held-out train + reconstructed metric (monotone DOWN)

print(f"{'config':>6} {'LB':>8} {'offline':>9}")
for c, l, o in zip(cfg, lb, offline): print(f"{c:>6} {l:>8.3f} {o:>9.4f}")
rho, _ = spearmanr(lb, offline)
print(f"\nSpearman rank correlation (LB vs offline): {rho:+.2f}   <-- should be +1 if offline were trustworthy")

fig, ax = plt.subplots(figsize=(5.2,4.2))
ax.scatter(lb, offline, s=90)
for c,l,o in zip(cfg,lb,offline): ax.annotate(f" {c}", (l,o))
ax.set_xlabel("public LB score (higher = better)"); ax.set_ylabel("offline edge_jaccard (reconstructed metric)")
ax.set_title(f"Offline vs LB: rank correlation {rho:+.2f} (inverted)")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()


## 4. Why — and what to do

Three compounding reasons the offline signal is untrustworthy here:
1. **Held-out train ≠ hidden test.** The visible clips are placeholders (§1); a train hold-out is *in-distribution*
   (same 2 embryos), so scores are optimistic and don't track the substituted test.
2. **The reconstructed metric ≠ the host metric.** Without the official scorer, everyone reverse-engineers
   `adj_edge_jaccard`; for these post-proc knobs it's not just optimistic, it's **directionally wrong** here.
3. **Sparse GT (§2)** makes edge-jaccard high-variance on top of that.

**Practical takeaway:** don't use offline scores to *rank* post-processing configs — you may pick the exact
opposite of what helps. Treat the **public LB (5/day) as the arbiter**: change one thing per submission, and
budget against public-LB overfitting (the hidden private set is what counts).

*Caveat / open question:* this uses one reconstructed metric + one hold-out. If anyone has the host's exact
metric, or a local scheme that actually correlates with the LB, I'd genuinely love to see it — that would be
worth more than any single config.
